In [10]:
import numpy as np
import pandas as pd
from pathlib import Path

from more_itertools import chunked

base_dir = Path('synthesibility_results')
model_paths_dict = {
    'SCENT\n(C+D+P)': 'rgfn_new_filtered/seh/dyn_lib_reward_exploitation_pen_none',
    'RGFN': 'rgfn_new_filtered/seh/rgfn_old',
}

In [22]:
from rgfn.gfns.reaction_gfn.proxies.path_cost_proxy import PathCostProxy
from rgfn import ReactionDataFactory
from notebooks.utils.common import get_path_cost_proxy, get_path_costs
from notebooks.utils.training_results import TrainingResults

path_cost_proxy = PathCostProxy(
            data_factory=ReactionDataFactory(
                reaction_path="../../data/rgfn_new_filtered/templates.txt",
                fragment_path="../../data/rgfn_new_filtered/fragments.txt",
                cost_path=f"../../data/rgfn_new_filtered/fragment_to_real_cost.json",
                yield_path="../../data/rgfn_new_filtered/templates_yields.csv",
            )
        )

for model, model_path in model_paths_dict.items():
    for seed in range(3):
        df = pd.read_csv(base_dir / model_path / f'sampling/OUT_molecules_{seed}_augmented.csv')
        path_df = pd.read_csv(f'../results/{model_path}/sampling/paths_{seed}.csv').iloc[:1000]
        path_df['paths'] = path_df['path'].apply(lambda x: eval(x))
        training_results = TrainingResults(model_name=model_path.split('/')[-1], templates_name='rgfn_new_filtered', results_dir=Path('../results'),
                                                   task_name='seh', threshold=8.0, seed=seed, include_paths=True)
        costs, _ = get_path_costs(path_df, path_cost_proxy, training_results.molecule_to_cheapest_cost)
        df['cost'] = costs
        df.to_csv(base_dir / model_path / f'sampling/OUT_molecules_{seed}_augmented_cost.csv', index=False)

Using 418 fragments, 112 reactions, and 206 anchored reactions
Cost mean and variance (4.699771268647007, 19.565368789313645)
Cost max 195.6361


path_costs: 100%|██████████| 1000/1000 [00:00<00:00, 8807.31it/s]
